In [1]:
import numpy as np
import pandas as pd

In [2]:
'''Tune These As Need Be'''

INITIAL_POPULATION_SIZE = 1_000_000
NUMBER_OF_GENES = 7
POPULATION_MIN = np.array([10.56,2098.43,-123.45,234,125.7,34.01,-12.45]) # Size should be NUMBER_OF_GENES
POPULATION_MAX = np.array([102.23,4045.12,-2.34,235,1231.2,132,0.782]) # Size should be NUMBER_OF_GENES, All values should be more than min
assert (((POPULATION_MIN).shape[0] == NUMBER_OF_GENES) and ((POPULATION_MAX).shape[0] == NUMBER_OF_GENES)), "Length Invalid"

In [3]:
rng = np.random.default_rng()

In [4]:
'''Define Sampling Distribution Here'''

POPULATION = rng.uniform(
    low=POPULATION_MIN,
    high=POPULATION_MAX,
    size=(INITIAL_POPULATION_SIZE, NUMBER_OF_GENES)
)

In [5]:
df = pd.DataFrame(POPULATION,columns=[f"Gene-{i+1}" for i in range(NUMBER_OF_GENES)])
print(df)

           Gene-1       Gene-2      Gene-3      Gene-4       Gene-5  \
0       88.833800  3137.676425 -109.909787  234.266395  1079.987543   
1       68.821933  2361.742693  -93.153788  234.117644   658.152220   
2       50.063831  3928.210563  -69.471675  234.129542   175.428634   
3       87.791791  3839.689963  -10.662676  234.191925   821.573343   
4       56.097741  2625.733266  -23.007267  234.659388   985.133679   
...           ...          ...         ...         ...          ...   
999995  16.827992  2163.640725  -73.681834  234.542829   757.246982   
999996  54.873030  2964.461781  -97.595187  234.821953   811.113846   
999997  87.298775  3105.196930  -93.087237  234.300602   285.183435   
999998  99.953718  2284.701716  -37.353705  234.013071  1174.975576   
999999  74.699885  3501.706022  -62.740404  234.582511   839.066912   

            Gene-6     Gene-7  
0       113.638830  -5.273357  
1        80.071397  -3.827854  
2        45.209236   0.381631  
3       110.847109 

In [6]:
'''Define Finction Here to Maximize, Ensure Fitness is allways Positive, Add a constant just incase for functions'''

# Current func = Sum of Squares

def fitness(chromosome):
    return np.sum(chromosome**2)

In [7]:
# Finding Best-Fit Chromosome of the Population

Best_Fit = np.argmax(np.apply_along_axis(fitness,1,POPULATION),axis=0)

In [8]:
print("Initial Best Fit\n",df.iloc[Best_Fit])

Initial Best Fit
 Gene-1      66.112280
Gene-2    4043.246126
Gene-3    -110.919241
Gene-4     234.933483
Gene-5    1230.604256
Gene-6     124.029519
Gene-7      -5.815431
Name: 577229, dtype: float64


In [9]:
'''
This method ensures that mixing occurs and all values stays withing the pre-defined interval.
'''

def crossover(chromosome1,chromosome2,alpha=0.7):
    if not (0 <= alpha <= 1): raise ValueError
    return alpha*chromosome1 + (1-alpha)*chromosome2

In [10]:
def mutation(chromosome,*,min_,max_):
    mutator = np.random.random(size=chromosome.shape)
    return np.clip(chromosome+mutator,min_,max_)

In [11]:
PROBA_CROSSOVER = 0.7
PROBA_MUTATION = 0.1

In [12]:
def evolve(Population, *, epochs=100, stagnation_limit=10):
    global fitness,mutation,crossover, PROBA_CROSSOVER, PROBA_MUTATION, INITIAL_POPULATION_SIZE, NUMBER_OF_GENES, POPULATION_MIN, POPULATION_MAX
    epochs_done = 0
    
    best_fitness_so_far = -np.inf
    stagnation = 0
    while (epochs_done < epochs) and (stagnation < stagnation_limit):
        epochs_done += 1
        # Roulette Wheel Selection
        weights = np.apply_along_axis(fitness, 1, Population)

        elite_index = np.argmax(weights)
        elite = Population[elite_index].copy()
        elite_fitness = weights[elite_index]
        
        weights /= np.sum(weights)
        
        parent_indices = np.random.choice(
            len(Population),
            size=len(Population),
            replace=True,
            p=weights
        )    
        parents = Population[parent_indices]

        # CrossOver
        children = []

        assert len(Population) % 2 == 0, "Population size must be even."
        for i in range(0, len(parents), 2):
            p1 = parents[i]
            p2 = parents[i + 1]
            if np.random.random() < PROBA_CROSSOVER:
                c1 = crossover(p1.copy(), p2.copy())
                c2 = crossover(p2.copy(), p1.copy())
            else:
                c1 = p1.copy()
                c2 = p2.copy()

            children.extend([c1, c2])

        children = np.array(children)

        # Mutation
        for i in range(len(children)):
            if rng.random() < PROBA_MUTATION:
                children[i] = mutation(children[i],min_=POPULATION_MIN,max_=POPULATION_MAX)

        # Fitness Calculations
        weights = np.apply_along_axis(fitness, 1, children)

        worst_index = np.argmin(weights)

        children[worst_index] = elite
        
        weights[worst_index] = elite_fitness
        
        best = weights.max()
        
        if best > best_fitness_so_far:
            best_fitness_so_far = best
            stagnation = 0
        else:
            stagnation += 1

        Population = children
        print(f"Epochs Done : {epochs_done}, BEST Fitness : {best}")
    
    if stagnation >= stagnation_limit:
        print("Stopped due to stagnation.")
    else:
        print("Maximum epochs reached.")
    return Population

In [16]:
Evolved_Population = evolve(POPULATION,epochs=200,stagnation_limit=30)

Epochs Done : 1, BEST Fitness : 17949510.86478744
Epochs Done : 2, BEST Fitness : 17949510.86478744
Epochs Done : 3, BEST Fitness : 17949510.86478744
Epochs Done : 4, BEST Fitness : 17949510.86478744
Epochs Done : 5, BEST Fitness : 17949510.86478744
Epochs Done : 6, BEST Fitness : 17952055.996249948
Epochs Done : 7, BEST Fitness : 17956791.243128344
Epochs Done : 8, BEST Fitness : 17956791.243128344
Epochs Done : 9, BEST Fitness : 17956791.243128344
Epochs Done : 10, BEST Fitness : 17956791.243128344
Epochs Done : 11, BEST Fitness : 17956791.243128344
Epochs Done : 12, BEST Fitness : 17956791.243128344
Epochs Done : 13, BEST Fitness : 17956791.243128344
Epochs Done : 14, BEST Fitness : 17956791.243128344
Epochs Done : 15, BEST Fitness : 17956791.243128344
Epochs Done : 16, BEST Fitness : 17956791.243128344
Epochs Done : 17, BEST Fitness : 17956791.243128344
Epochs Done : 18, BEST Fitness : 17956791.243128344
Epochs Done : 19, BEST Fitness : 17956791.243128344
Epochs Done : 20, BEST Fit

In [17]:
df = pd.DataFrame(Evolved_Population,columns=[f"Gene-{i+1}" for i in range(NUMBER_OF_GENES)])

In [18]:
print("Discovered Best Fit\n",df.iloc[np.argmax(np.apply_along_axis(fitness,1,Evolved_Population),axis=0)])

Discovered Best Fit
 Gene-1      69.443799
Gene-2    4045.120000
Gene-3    -108.558833
Gene-4     235.000000
Gene-5    1231.200000
Gene-6     126.327515
Gene-7      -3.753930
Name: 869724, dtype: float64


As we can See, the GA reached a better result than original, But this is not the actual Optimum.

This is why multiple Simutaions are needed with multiple Initial Conditions and Corossover + Mutation Probabilites.

Please Refer to Notebook - https://www.kaggle.com/code/animeguylrn/evolution-simulator-genetic-algorithms

It is an extension of this notebook with a Simulator Class to automate such simmulations.